## Understanding the dataset

In [22]:
import pandas as pd

df = pd.read_csv("customer_support_tickets.csv")
print(df.head())

   Ticket ID        Customer Name              Customer Email  Customer Age  \
0          1        Marisa Obrien  carrollallison@example.com            32   
1          2         Jessica Rios    clarkeashley@example.com            42   
2          3  Christopher Robbins   gonzalestracy@example.com            48   
3          4     Christina Dillon    bradleyolson@example.org            27   
4          5    Alexander Carroll     bradleymark@example.com            67   

  Customer Gender Product Purchased Date of Purchase      Ticket Type  \
0           Other        GoPro Hero       2021-03-22  Technical issue   
1          Female       LG Smart TV       2021-05-22  Technical issue   
2           Other          Dell XPS       2020-07-14  Technical issue   
3          Female  Microsoft Office       2020-11-13  Billing inquiry   
4          Female  Autodesk AutoCAD       2020-02-04  Billing inquiry   

             Ticket Subject  \
0             Product setup   
1  Peripheral compatibil

In [23]:
print(df.columns)

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')


## Import Libraries

In [24]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

## Remove missing values

In [25]:
df = df.dropna()  

## Text Preprocessing

In [26]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isnull(text):
        return ""
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [word for word in words if word not in stop_words and len(word) > 2]
    
    return " ".join(words)

# Combine Subject + Description
df['combined_text'] = df['Ticket Subject'] + " " + df['Ticket Description']

# Clean text
df['clean_text'] = df['combined_text'].apply(clean_text)

## Convert Text → Numbers (TF-IDF)

In [27]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])  # TF-IDF converts text into numerical form by giving importance to meaningful words.

## CATEGORY CLASSIFICATION MODEL

In [28]:
y_category = df['Ticket Type']   

X_train, X_test, y_train, y_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42
)

model_cat = LogisticRegression(max_iter=200)
model_cat.fit(X_train, y_train)

y_pred_cat = model_cat.predict(X_test)

print("Category Model Performance:")
print(classification_report(y_test, y_pred_cat)) # Precision measures correctness, recall measures coverage, and F1-score balances both.

Category Model Performance:
                      precision    recall  f1-score   support

     Billing inquiry       0.29      0.20      0.23       123
Cancellation request       0.16      0.17      0.16        96
     Product inquiry       0.24      0.20      0.21       102
      Refund request       0.24      0.27      0.25       122
     Technical issue       0.22      0.30      0.25       111

            accuracy                           0.23       554
           macro avg       0.23      0.23      0.22       554
        weighted avg       0.23      0.23      0.23       554



## PRIORITY MODEL

In [29]:
y_priority = df['Ticket Priority'] 

X_train, X_test, y_train, y_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42
)

model_pri = LogisticRegression(max_iter=200)
model_pri.fit(X_train, y_train)

y_pred_pri = model_pri.predict(X_test)

print("Priority Model Performance:")
print(classification_report(y_test, y_pred_pri)) # Precision measures correctness, recall measures coverage, and F1-score balances both.
# CLASS-WISE PERFORMANCE: It shows performance for each class separately High, Medium and Low

Priority Model Performance:
              precision    recall  f1-score   support

    Critical       0.31      0.31      0.31       156
        High       0.24      0.25      0.25       130
         Low       0.28      0.25      0.26       133
      Medium       0.30      0.31      0.30       135

    accuracy                           0.28       554
   macro avg       0.28      0.28      0.28       554
weighted avg       0.28      0.28      0.28       554



## CONFUSION MATRIX

In [30]:
print(confusion_matrix(y_test, y_pred_pri))

[[49 40 32 35]
 [38 33 28 31]
 [36 31 33 33]
 [34 34 25 42]]


## FINAL PREDICTION SYSTEM

In [31]:
def predict_ticket(text):
    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])
    
    category = model_cat.predict(vector)[0]
    priority = model_pri.predict(vector)[0]
    
    return category, priority

## Test

In [32]:
print(predict_ticket("My account is locked, please fix this immediately"))

('Technical issue', 'Low')
